# Demand Forecast Solution

## Data Analysis

In [1]:
import pandas as pd

In [83]:
train_path = '/home/harish/Documents/demand_forecast/data/train.csv'
test_path = '/home/harish/Documents/demand_forecast/data/test.csv'

In [84]:
train_df = pd.read_csv(train_path, parse_dates=['date'])
test_df = pd.read_csv(train_path, parse_dates=['date'])

In [4]:
train_df.info(), test_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 913000 entries, 0 to 912999
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype         
---  ------  --------------   -----         
 0   date    913000 non-null  datetime64[us]
 1   store   913000 non-null  int64         
 2   item    913000 non-null  int64         
 3   sales   913000 non-null  int64         
dtypes: datetime64[us](1), int64(3)
memory usage: 27.9 MB
<class 'pandas.DataFrame'>
RangeIndex: 913000 entries, 0 to 912999
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype         
---  ------  --------------   -----         
 0   date    913000 non-null  datetime64[us]
 1   store   913000 non-null  int64         
 2   item    913000 non-null  int64         
 3   sales   913000 non-null  int64         
dtypes: datetime64[us](1), int64(3)
memory usage: 27.9 MB


(None, None)

In [5]:
train_df.isnull().sum(), test_df.isnull().sum()

(date     0
 store    0
 item     0
 sales    0
 dtype: int64,
 date     0
 store    0
 item     0
 sales    0
 dtype: int64)

In [6]:
train_df.item.nunique(), train_df.store.nunique()

(50, 10)

In [13]:
train_df.item.unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50])

In [7]:
train_df.loc[(train_df['item'] == 1) & (train_df['store'] == 1)].sort_values('date', ascending=True)

,date,store,item,sales
0,2013-01-01,1,1,13
1,2013-01-02,1,1,11
2,2013-01-03,1,1,14
3,2013-01-04,1,1,13
4,2013-01-05,1,1,10
...,...,...,...,...
1821,2017-12-27,1,1,14
1822,2017-12-28,1,1,19
1823,2017-12-29,1,1,15
1824,2017-12-30,1,1,27


### Sales Trends on some Items

In [8]:
import plotly.express as px

In [24]:
# Item : 1 & Store 1
df_1_1 = train_df.loc[(train_df['item'] == 1) & (train_df['store'] == 1)].sort_values('date', ascending=True)
print(f"Min date : {df_1_1['date'].min()}, Max date: {df_1_1['date'].max()}")

px.line(df_1_1, x='date', y='sales', title='Item: 1, Store: 1')

Min date : 2013-01-01 00:00:00, Max date: 2017-12-31 00:00:00


In [23]:
# Item : 2 & Store 1
df_2_1 = train_df.loc[(train_df['item'] == 2) & (train_df['store'] == 1)].sort_values('date', ascending=True)
print(f"Min date : {df_2_1['date'].min()}, Max date: {df_2_1['date'].max()}")

px.line(df_2_1, x='date', y='sales')

Min date : 2013-01-01 00:00:00, Max date: 2017-12-31 00:00:00


In [25]:
# Item : 3 & Store 6
df_3_6 = train_df.loc[(train_df['item'] == 3) & (train_df['store'] == 6)].sort_values('date', ascending=True)
print(f"Min date : {df_3_6['date'].min()}, Max date: {df_3_6['date'].max()}")
px.line(df_3_6, x='date', y='sales', title='Item: 3, Store: 6').show()

Min date : 2013-01-01 00:00:00, Max date: 2017-12-31 00:00:00


### Verdict:
- Time series are **Stationary** and almost constant mean, variance and seasonality pattern observed.
- By observing line plots, I am observing these products are yearly seasonal. Patterns are repeating on a yearly pattern.
- We can proceed further to build 2 models i.e. AutoARIMA for Statistical model and Random Forest Regressor for Machine learning model

## Data Preprocessing for Modelling

In [163]:
class FeatureEngineeringPipeline:
    def __init__(self, data_path: str):
        self.data = pd.read_csv(data_path, parse_dates=['date'])
     
    def encode_store_item(self):
        self.data['item_store'] = self.data['item'].astype(str) + '_' + self.data['store'].astype(str)
        self.data.rename(columns={'item_store': 'unique_id', 'date': 'ds', 'sales': 'y'}, inplace=True)

    def extract_date_features(self):
        self.data['year'] = self.data['ds'].dt.year
        self.data['month'] = self.data['ds'].dt.month
        self.data['day_of_week'] = self.data['ds'].dt.day_of_week # 0 = monday, 6 = Sunday

    def make_lags(self):
        self.data['lag_1'] = self.data.groupby('unique_id')['y'].shift(1)
        self.data['lag_7'] = self.data.groupby('unique_id')['y'].shift(7)

    def add_moving_average(self):
        return
        self.data['ma'] = self.data.groupby('unique_id')['y'].rolling(7).sum().reset_index()
    def fill_missing_dates(self):
        """ 
        - Although data is cleaned enough and we don't need to fill the missing values.
        - Since, we've introduced lags, we got missing values.
        """
        self.data.fillna(0, inplace=True)
    def trigger(self, test=False):
        if test == False:
            self.encode_store_item()
            self.extract_date_features()
            self.make_lags()
            self.add_moving_average()
            self.fill_missing_dates()
        else:
            self.data['item_store'] = self.data['item'].astype(str) + '_' + self.data['store'].astype(str)
            self.data.rename(columns={'item_store': 'unique_id', 'date': 'ds'}, inplace=True)
            self.data.drop('id', inplace=True, axis = 1)
        return self.data

Below is the way to invoke the FeatureEngineeringPipeline, which we'll invoke sequentially before training.

In [105]:
# preprocess = FeatureEngineeringPipeline(data_path='/home/harish/Documents/demand_forecast/data/train.csv').trigger()
# preprocess.head(10)

### Verdict:
- As per the documentation, I calculated lags, moving averages, encode item store into a unified id to train model on each paired SKU.
- We can make use of features as a **Confounding Variables**.
- I'll be using statsforecast for AutoARIMA and RandomForest from ScikitLearn.

## Modelling / Training

### 1. AutoARIMA

In [164]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, SeasonalNaive

In [165]:
class Training:
    def __init__(self):
        self.models = [AutoARIMA(), SeasonalNaive(season_length=365)]
        self.horizon = 90
        self.freq = 'D'
    def build_sf_model(self):
        self.sf = StatsForecast(models=self.models, freq=self.freq, n_jobs=-1)
    def forecast_(self, preprocessed_data, test=False):
        if test == False:
            self.df = preprocessed_data[['unique_id', 'ds', 'y']]
            self.build_sf_model()
            return self.sf.cross_validation(h=self.horizon, df=self.df, step_size=7, n_windows=3)
        else:
            return self.sf.forecast(h=self.horizon, df=self.df)


In [166]:
preprocessed_data = FeatureEngineeringPipeline(data_path='/home/harish/Documents/demand_forecast/data/train.csv').trigger()
training_instance = Training()
training_predictions = training_instance.forecast_(preprocessed_data=preprocessed_data)

#### Evaluate through metrics

In [168]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

def calculate_cv_metrics(y_true, y_pred, model_name):
    import numpy as np
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    valid_mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    y_true_clean = y_true[valid_mask]
    y_pred_clean = y_pred[valid_mask]

    if len(y_true_clean) == 0:
        return {'model': model_name, "error": "No data"}
    
    residuals = y_true_clean - y_pred_clean

    metrics = {
        "model": model_name,
        "mae": mean_absolute_error(y_true_clean, y_pred_clean),
        "rmse": np.sqrt(mean_squared_error(y_true_clean, y_pred_clean)),
        "mape": mean_absolute_percentage_error(y_true_clean, y_pred_clean)
    }
    return metrics


def calculate_window_metrics(cv_results_df, step_size=7, n_windows=3):
    models = ['AutoARIMA']
    window_metrics = []
    for window in range(n_windows):
        window_data = {
            "window": window,
            "step_size": step_size
        }
        for model in models:
            metrics = calculate_cv_metrics(cv_results_df['y'], cv_results_df[model], model_name=model)
            for key, value in metrics.items():
                if key != "model":
                    window_data[f"{model}_{key}"] = value
        window_metrics.append(window_data)
    return pd.DataFrame(window_metrics)

metrics = calculate_window_metrics(cv_results_df=training_predictions)
print(metrics)

   window  step_size  AutoARIMA_mae  AutoARIMA_rmse  AutoARIMA_mape
0       0          7      11.020567        14.60407        0.246079
1       1          7      11.020567        14.60407        0.246079
2       2          7      11.020567        14.60407        0.246079


### Test Predictions

In [169]:
preprocessed_dataT = FeatureEngineeringPipeline(data_path='/home/harish/Documents/demand_forecast/data/test.csv').trigger(test=True)
test_predictions = training_instance.forecast_(preprocessed_data=preprocessed_dataT, test=True)

In [170]:
test_predictions

,unique_id,ds,AutoARIMA,SeasonalNaive
0,10_1,2018-01-01,61.225983,53.0
1,10_1,2018-01-02,57.409576,51.0
2,10_1,2018-01-03,57.740662,53.0
3,10_1,2018-01-04,63.718842,46.0
4,10_1,2018-01-05,61.830669,46.0
...,...,...,...,...
44995,9_9,2018-03-27,48.733212,49.0
44996,9_9,2018-03-28,48.733212,52.0
44997,9_9,2018-03-29,48.733212,52.0
44998,9_9,2018-03-30,48.733212,55.0


### Verdict:
- There's a lot more to it to improve the model's performance.
- Due to time being, I am able to work on AutoARIMA to its fullest.
- Got RMSE to 14 and MAPE to around 24%

## Submission

In [171]:
submission_df = pd.read_csv('/home/harish/Documents/demand_forecast/data/sample_submission.csv')

In [173]:
submission_df['sales'] = test_predictions['AutoARIMA']

In [175]:
submission_df.to_csv('FinalSubmission.csv', index=False)